In [38]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD
from dateparser.search import search_dates
import dateparser, re
import json

# Entity Alignment

In [ ]:
ner = pipeline("ner", model="Davlan/xlm-roberta-base-ner-hrl", grouped_entities=True)
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
EX = Namespace("")


SCALE_WORDS = [
    ("billion", 1e9), ("milliarden", 1e9), ("milliarde", 1e9),
    ("million", 1e6), ("millionen", 1e6),
    ("thousand", 1e3), ("tausend", 1e3),
]

CURRENCY_WORDS = ["$", "€", "eur", "euro", "dollar", "usd", "eur.", "usd."]

def normalize_money_phrase(text: str):
    """Convert money phrase to base numeric units if possible."""
    t = text.lower().replace(" ", "").replace(" ", " ")
    # Replace decimal commas for DE style numbers
    t = t.replace(",", ".")
    # Find a number (possibly with decimals)
    nm = re.search(r"(\d+(?:\.\d+)?)", t)
    if not nm:
        return None
    val = float(nm.group(1))
    # Scale by magnitude words
    for w, scale in SCALE_WORDS:
        if w in t:
            val *= scale
            break
    return val

def extract_money_values(text: str):
    """Find money-like phrases then normalize them."""
    # Capture things like "$2.3 billion", "3 Milliarden Dollar", "€750 million", "USD 1.2 million"
    pattern = r"(?:[$€]\s?\d[\d\.,]*\s?(?:\w+)?)|(?:\d[\d\.,]*\s?(?:billion|million|thousand|milliarden|milliarde|millionen|tausend)\s?(?:dollar|eur|euro|usd)?)"
    matches = re.findall(pattern, text, flags=re.IGNORECASE)
    vals = []
    for m in matches:
        v = normalize_money_phrase(m)
        if v:
            vals.append(v)
    return vals

def extract_dates(text: str):
    """Return ISO dates found in the text (en+de)."""
    out = []
    res = search_dates(text, languages=["en", "de"])
    if res:
        for _, dt in res:
            try:
                out.append(dt.date().isoformat())
            except Exception:
                pass
    # Deduplicate, keep order
    seen, uniq = set(), []
    for d in out:
        if d not in seen:
            uniq.append(d)
            seen.add(d)
    return uniq

def build_kg(text: str):
    g = Graph()
    g.bind("ex", EX)
    stmt = URIRef(EX[f"Statement_{abs(hash(text))}"])
    g.add((stmt, RDF.type, EX.Statement))

    # 1) NER for ORG/PER/LOC/GPE
    for ent in ner(text):
        etype = ent["entity_group"]
        value = ent["word"].strip()
        if not value:
            continue
        node = URIRef(EX[value.replace(" ", "_")])
        g.add((node, RDFS.label, Literal(value)))
        g.add((node, RDF.type, EX[etype]))

        if etype in ["LOC", "GPE"]:
            g.add((stmt, EX.location, node))
        elif etype == "ORG":
            g.add((stmt, EX.organization, node))
        elif etype == "PERSON":
            g.add((stmt, EX.person, node))
        # (We do NOT rely on NER for DATE/MONEY)

    # 2) Fallback: dates (robust across languages)
    dates = extract_dates(text)
    if dates:
        # Choose first date mentioned (you can store all if you like)
        g.add((stmt, EX.date, Literal(dates[0], datatype=XSD.date)))

    # 3) Fallback: monetary amounts
    money_vals = extract_money_values(text)
    if money_vals:
        # If multiple, you can store the largest or the first; here we take the first
        g.add((stmt, EX.amount, Literal(money_vals[0], datatype=XSD.decimal)))

    return g

def extract_labels(g: Graph):
    return {s: str(o) for s,_,o in g.triples((None, RDFS.label, None))}

def align_entities(labels1, labels2, threshold=0.75):
    emb1 = {n: embedder.encode(lbl, convert_to_tensor=True) for n,lbl in labels1.items()}
    emb2 = {n: embedder.encode(lbl, convert_to_tensor=True) for n,lbl in labels2.items()}
    alignments = {}
    for n1, e1 in emb1.items():
        best, best_score = None, -1
        for n2, e2 in emb2.items():
            score = float(util.cos_sim(e1, e2))
            if score > best_score:
                best, best_score = n2, score
        if best_score >= threshold:
            alignments[n1] = (best, best_score)
    return alignments

def get_literal(g, subj, pred):
    obj = next((o for _,_,o in g.triples((subj, pred, None))), None)
    return obj.toPython() if obj else None

def compare_facts(g1: Graph, g2: Graph, rel_tol=0.05):
    stmt1 = next(s for s,_p,_o in g1.triples((None, RDF.type, EX.Statement)))
    stmt2 = next(s for s,_p,_o in g2.triples((None, RDF.type, EX.Statement)))

    val1, val2 = get_literal(g1, stmt1, EX.amount), get_literal(g2, stmt2, EX.amount)
    date1, date2 = get_literal(g1, stmt1, EX.date), get_literal(g2, stmt2, EX.date)
    loc1, loc2 = get_literal(g1, stmt1, EX.location), get_literal(g2, stmt2, EX.location)
    org1, org2 = get_literal(g1, stmt1, EX.organization), get_literal(g2, stmt2, EX.organization)
    per1, per2 = get_literal(g1, stmt1, EX.person), get_literal(g2, stmt2, EX.person)

    print("\n=== FACT COMPARISON ===")

    # Money
    if val1 is not None and val2 is not None:
        diff = abs(val1 - val2) / max(val1, val2)
        print(f"💶 Money: {val1:.2e} ↔ {val2:.2e} → {'✅ match' if diff <= rel_tol else '❌ mismatch'}")
    else:
        print("💶 Money: Not found in one or both sentences.")

    # Date
    if date1 and date2:
        print(f"📅 Date: {date1} ↔ {date2} → {'✅ match' if date1 == date2 else '❌ mismatch'}")
    else:
        print("📅 Date: Not found in one or both sentences.")

    # Location (string compare on resource IRI tail; for stronger checks, align with embeddings)
    if loc1 and loc2:
        l1 = loc1.split("#")[-1]
        l2 = loc2.split("#")[-1]
        print(f"🌍 Location: {l1} ↔ {l2} → {'✅ match' if l1 == l2 else '❌ mismatch'}")
    else:
        print("🌍 Location: Not found in one or both sentences.")

    # Organization
    if org1 and org2:
        o1 = org1.split("#")[-1]; o2 = org2.split("#")[-1]
        print(f"🏢 Organization: {o1} ↔ {o2} → {'✅ match' if o1 == o2 else '❌ mismatch'}")

    # Person
    if per1 and per2:
        p1 = per1.split("#")[-1]; p2 = per2.split("#")[-1]
        print(f"👤 Person: {p1} ↔ {p2} → {'✅ match' if p1 == p2 else '❌ mismatch'}")

def analyze(text_en, text_de):

    g_en = build_kg(text_en)
    g_de = build_kg(text_de)

    labels_en = extract_labels(g_en)
    labels_de = extract_labels(g_de)
    alignments = align_entities(labels_en, labels_de)

    print("\n=== ENTITY ALIGNMENTS (labels) ===")
    for a,(b,score) in alignments.items():
        print(f"- {labels_en[a]} ↔ {labels_de[b]} (sim={score:.2f})")

    compare_facts(g_en, g_de)

Device set to use cpu
c:\Users\erenc\miniconda3\envs\CV2\Lib\site-packages\transformers\pipelines\token_classification.py:186: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


# Evaluation

In [40]:
import json

en_data = json.load(open("data/eval/eval_sample_en.json", "r", encoding="utf-8"))[0]['para']
de_data = json.load(open("data/eval/eval_sample_de.json", "r", encoding="utf-8"))[0]['para']

for i, (en, de) in enumerate(zip(en_data, de_data)):
    print(f"TEST {i}")
    analyze(en['para'], de['para'])

TEST 0

=== ENTITY ALIGNMENTS (labels) ===

=== FACT COMPARISON ===
💶 Money: Not found in one or both sentences.
📅 Date: 2025-10-29 ↔ 2025-10-29 → ✅ match
🌍 Location: Not found in one or both sentences.
TEST 1


C:\Users\erenc\AppData\Local\Temp\ipykernel_6724\2596555616.py:46: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  res = search_dates(text, languages=["en", "de"])



=== ENTITY ALIGNMENTS (labels) ===

=== FACT COMPARISON ===
💶 Money: Not found in one or both sentences.
📅 Date: 2025-03-18 ↔ 2025-03-18 → ✅ match
🌍 Location: Not found in one or both sentences.
TEST 2

=== ENTITY ALIGNMENTS (labels) ===
- Republic of Moldova ↔ Moldau (sim=0.90)

=== FACT COMPARISON ===
💶 Money: Not found in one or both sentences.
📅 Date: Not found in one or both sentences.
🌍 Location: Republic_of_Moldova ↔ Moldau → ❌ mismatch
TEST 3

=== ENTITY ALIGNMENTS (labels) ===
- EUROPEAN UNION ↔ EUROPÄISCHEN UNION (sim=0.92)

=== FACT COMPARISON ===
💶 Money: Not found in one or both sentences.
📅 Date: Not found in one or both sentences.
🌍 Location: Not found in one or both sentences.
🏢 Organization: EUROPEAN_UNION ↔ EUROPÄISCHE_PARLAMENT → ❌ mismatch
TEST 4

=== ENTITY ALIGNMENTS (labels) ===

=== FACT COMPARISON ===
💶 Money: Not found in one or both sentences.
📅 Date: Not found in one or both sentences.
🌍 Location: Not found in one or both sentences.
🏢 Organization: Union ↔ 

KeyboardInterrupt: 